# 04 · Training: one model on the train split, written as a numbered run

**What it does.** Fits one model from the feature set in force and writes it as a numbered
**run**, `<roots.data>/<use case>/06_models/r001/`, `r002/`, …. Every choice is made on the train
split's clients only: which features stay (automated selection), the family and its
configuration (the search) and the calibration. No test or validation row enters a fit or a
choice, so the numbers here are train-only; the first honest number is the test split's, in
`05_evaluation`.

**How one run goes:**
1. The gate, then the feature set in force: the candidates minus the excluded and the held
   features.
2. The train frame is collected with its client folds, its counts per period and its column
   types are checked against EDA's, and the encoder is fitted on it.
3. The run number is reserved.
4. Automated selection: the cluster rule, the floor, importance pruning.
5. One search per family on its share of the trial budget, every trial judged on the client
   folds; the best family and configuration win.
6. The fold models at that configuration give every train row an out-of-fold score; the
   calibration is fitted on those.
7. The final model is fitted on every train row, written as the bundle and checked on reload.

**Before you start: the gate.** No model without data knowledge: Training refuses unless
`02_eda` and `03_feature_checks` ran on this data definition, with the settings and sources in
force now. The message says why and names the fix:
- **no EDA or Feature Checks** for this data definition: run `02_eda` and `03_feature_checks`
  (step 1 of each), or the CLI `python -m core_modelling run conf/base/cc_acquisition.yml
  --stages eda feature_checks training`;
- **EDA or Feature Checks ran with other settings**, or the environment file now maps a source
  elsewhere: rerun both, in that order;
- **the panel's counts per period, or a column's type, differ from EDA's**: a source changed
  after EDA read it; rerun both.

A refused run takes no run number and writes nothing. For the other example, change `CONFIG` in
the setup cell.

**Automation levels** (`automation_level`; this example is `assisted`): Training runs the same
at both; only the default trial budget differs (`training.max_iter`: 60 at `autonomous`, 200 at
`assisted`). At `autonomous`, `python -m core_modelling run` first runs EDA and Feature Checks
when they are missing; `run_training` here never does. At both levels a person signs off every
feature of the model before it is registered (`05_evaluation`).

**What it writes**, in the run folder: `run.json` (what the run is built from, written first),
`bundle/` (the model file, `bundle.json`, `manifest.json`), `ledger_run.parquet` (the per-run
ledger), `stage2/` (the run's own copy of what it read from EDA and Feature Checks) and, last,
`training/` (`training.json`, `tables/`, the search's logs, `resolved_config.yml`); then a
`run` entry in the decision audit log. Training writes no report: the model results report is
Evaluation's.

**Docs:** `docs/components/training.md` · `docs/components/model_bundle.md` · next:
`05_evaluation`.

In [ ]:
from pathlib import Path

from IPython.display import HTML, display

import core_modelling
from core_modelling.config import load_config
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.runs import list_runs, show_run
from core_modelling.training import run_training

# The use case and where the framework runs, as absolute paths. The repo's examples are found
# from the installed package; a use-case repo writes its own config path here and may read the
# environment file from $CORE_MODELLING_ENVIRONMENT instead.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"  # or conf/base/td_acquisition.yml
ENTITY = "entity_a"
ENVIRONMENT = str(REPO / "conf/local/environment.yml")  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(ENVIRONMENT)
spark = get_spark_session(env)
cfg = load_config(CONFIG)
print(cfg.use_case, cfg.task, cfg.automation_level)
print(cfg.training)

## 1. Train
`run_training` applies the decision audit log to the config, passes the gate and trains;
`result.views` holds the views below, `result.tables` Training's tables and `result.meta`
`training.json`'s content. It prints, in order:
- the train frame's lines, ending with `collect train`: its rows × columns and its size in
  memory;
- `training r…`: the run number and the feature set in force (step 2), then any warning;
- `selection`: the features left after the three rules (step 4);
- `search <family>`: per family, its trials, seconds and best fold metric (step 3);
- `training r… written`: the run folder.

**The budget.** The default trial budget is a main run's. For a first look, record a smaller
`training.max_iter` (step 9) and run a quick one: its search table tells you how to spend the
main run's budget. With `training.categoricals.unordered: native`, the default, a CatBoost
trial costs several times a trial of the other families, and about three times its own with
`codes` for the same fold metric, so at an equal trial count CatBoost takes most of the
search's time. Read the quick run's `seconds_per_trial` first, then give `catboost` fewer
trials, or record `codes` (it applies to every family) when CatBoost's fold metric does not pay
for the time (step 9).

**When it fails** before `training r…` is printed, nothing is written: every refusal (the gate,
a changed panel, a constraint on an unordered categorical, a name in `training.search_space` a
family does not take, a trial budget smaller than the number of families) comes before the run
number is reserved. Fix what the message names and rerun the cell. A failure after it leaves the run folder without `training/`: an
incomplete run (step 8), whose number is never given again.

**The same config, audit log, data and seed give the same model**, with the same `model_hash`,
on the same machine and library versions; a rerun only takes the next run number. The
exception is `training.time_budget_s`: the search then stops on measured seconds, so only the
recipe repeats.

In [ ]:
result = run_training(cfg, ENTITY, env, spark)
views = result.views

## 2. The feature set in force and the warnings
**The feature set in force** is the candidates minus the target's lineage, the unsupported
columns, the features a person excluded and the **held** ones. A held feature has an open flag
from Feature Checks: for Training it is an excluded feature without a final decision, so the
model is built without it until a person keeps or excludes it. The line names the held, excluded
and kept features (a `keep` returns a held feature and protects it from selection), and how many
Feature Checks' run held.

On the card example, after `03_feature_checks` ran to its end, the three static attributes
are kept and `holding_cnt_trap_lookahead` is held (the walk-through excluded it and undid the
exclude).

**Check:** at `assisted`, a flag you settled in `03_feature_checks` is not in the held list; a
held feature you need is a `keep` (step 9).

**The warnings** show only those that apply; none changes what the run did:
- **Features Feature Checks held are in training with no decision**: a revert undid their flags.
  Rerun `03_feature_checks` step 1 to hold them again, or record a `keep` or an `exclude`.
- **Few converting train clients or positive rows**: the fold metric is noisy. Set
  `training.fold_repeats` to 3, keep the trial budget small, and read differences between runs
  as noise unless they are large.
- **`training.time_budget_s` is set**: the run repeats the recipe, not the model.
- **An ordered feature without a null in train**: a null or an unseen level at scoring is a value
  the model never saw. **A constrained feature with nulls while `catboost` is searched**: CatBoost
  counts a null as the smallest value; its fold loss carries the cost.
- **A constraint against EDA's direction**, **pruning undone**, **a wide level factor**: steps 7,
  4 and 5.

In [ ]:
display(HTML(views["opening"]), HTML(views["warnings"]))

## 3. What the search did
Selection ran before the search (step 4): every trial sees the final features only.
- **One search per family** in `training.estimator_list` (`lgbm`, `xgboost` leaf by leaf,
  `xgb_limitdepth` level by level, `catboost`), each on its own budget: `training.max_iter`
  split equally (the remainder to the first families), or `training.trials`, a count per family.
- **A trial** is one configuration fitted on every client fold (`training.folds`, 5 by default,
  times `training.fold_repeats` dealings); its **fold metric** is the mean over the folds. The
  metric follows the decision, fixed before any result: `log_loss` for a `probability` decision
  (this example), `roc_auc` for `ranking` or `top_n`, `rmse` for regression unless
  `decision.metric` names another.
- **The steps**: each family starts from a fixed configuration and moves four or five bounded
  parameters from there, keeping a step that improves the fold metric. The ranges are in
  `docs/components/training.md`; `training.search_space` overrides one.
- **The winner** is the family and configuration with the best fold metric over all families.
  The framework then fits it itself: the search tool only searches.

**The table**, one row per family: `trials`; `seconds` in total, `seconds_per_trial` (the
median) and `seconds_longest`; `best_fold_metric` and `best_trial`, the trial that first reached
it; `share_seconds`, the family's share of the search's time. The last table: the chosen
family and its parameters.

**The chart**: each family's best fold metric so far against its seconds. A line still
improving at its last trial wanted more trials; a line flat after a few trials had enough.
Families a hair apart are a tie: the noise between folds is larger than such a gap.

**Decide** (step 9):
- **The budget**: give the leading family more trials and a family that settled early or
  trails clearly fewer, with `training.trials`, from the seconds per trial here; raise
  `training.max_iter` when the leader still improves at its last trial.
- **The families**: dropping one from `training.estimator_list` saves its time; a single family
  fixes the choice.
- **`training.fold_repeats`**: 3 when converting train clients are few (under about 100, or 200
  positive rows): each trial is judged on three dealings, so the winner depends less on how the
  clients fell into folds. The search takes about three times as long.

In [ ]:
search = views["search"]
display(search["summary"], search["progress"], search["best"])

## 4. Selection
Automated selection runs at both levels, on the train split only, before the search. Three rules,
in this order, each switched in `training.selection`:
1. **The cluster rule** (`cluster`): from each redundancy cluster Feature Checks found (its
   near-duplicates), the member with the highest train concordance stays; the others are dropped
   and listed under it (`represented_by`).
2. **The floor** (`floor_z`, 2 by default): a feature whose train concordance is below
   0.5 + `floor_z` standard errors is dropped. The standard error shrinks as converting train
   clients grow, so with many events only features with no signal fall below it.
3. **Importance pruning** (`prune`): five columns of random numbers join the features, quick
   fold models are fitted, and a feature that contributes no more than the best noise column is
   dropped. If the out-of-fold loss rises without those features, nothing is pruned (`undone`).

A feature with a person's `keep` is measured by every rule and never dropped (`protected`). No
drop writes an audit-log entry: each run recomputes its own, and the per-run ledger records
them.

**The views:** a line with how many features entered and how many are in the final list; the
features per rule and outcome (`passed`, `dropped`, `protected`, `undone`; a rule after the one
that dropped a feature does not measure it); the dropped features, each with its rule's `value`
against its `threshold`.

**Decide** (step 9):
- **A dropped feature the model needs**: a `keep`; it then passes every rule.
- **`floor_z`**: lower it to 1, or set it to `null` (off), below about 300 converting train
  clients, or when the candidates are few and chosen by experts: at 2 a real but weak feature is
  then dropped more often than kept. Never derive it from the event count: decide it.
- **`cluster`, `prune`**: switch a rule off only with few, hand-picked candidates.

In [ ]:
selection = views["selection"]
shown = [selection["summary"], selection["counts"], *selection["dropped"]]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 5. Calibration and the out-of-fold numbers
**The fold models**: one per fold at the winning configuration, each fitted without that fold's
clients and scoring them, give every train row an **out-of-fold** margin (the score before
calibration; for a yes/no target, its log-odds). The calibration is fitted on these margins,
inside train; then the fold models are discarded.

**The method** (`training.calibration.method`) follows the decision unless a person sets it:
for a yes/no target `platt` for a `probability` decision, else `none`; for regression `mean` for
the `tweedie` and `poisson` objectives, else `none`.
- `platt`: the label regressed on the margin, two parameters; `a` = 1, `b` = 0 leaves the
  model's scores as they are.
- `beta`: three parameters on the probability, for a bend Platt cannot follow.
- `none`: the model's own score.
- `mean` (regression): the level factor, the total observed over the total predicted, with a
  client-bootstrap interval.

**The table**: for a yes/no target, every method's out-of-fold log loss (lower is better),
whatever the decision, the bundle's marked `chosen`: the cost of the choice in one column. For
regression: the factor, its interval and the rows holding half of the target's total (few rows:
a few clients decide the level). **The line**: the bundle's method; for a yes/no target, the
true and sampled rates and the **offset**, which undoes `negative_downsampling`
(`training.sampling`) exactly and is zero without sampling.

**The out-of-fold numbers**, the last table: `out_of_fold`, the decision's metric on
the calibrated out-of-fold scores; `out_of_fold_concordance` (the AUC for a yes/no target); and
`in_sample_concordance`, the final model on the rows it was fitted on. **They are optimistic:**
the features and the configuration were chosen on the same folds. In sample far above out of
fold means the model fits its own train clients much better than unseen ones; `05_evaluation`
sets both beside the test split.

**Decide:** `beta` clearly below `platt` in log loss is a `set` of
`training.calibration.method` to `beta`, then train again (step 9). Under sampling the log
losses are at the sampled rate: compare them with each other only.

In [ ]:
calibration = views["calibration"]
display(HTML(calibration["summary"]), calibration["methods"], calibration["metrics"])

## 6. The explanation
What the model learned, from the fold models' out-of-fold **contributions** on the explanation
sample (5,000 seeded train rows, or every row when train has fewer). A contribution is how far a
feature moved one row's margin from the base value: positive raises the score. A row's
contributions and the base value add up to its margin.
- **The importance bars**: each feature's share of the mean absolute contribution, the 15
  largest, the rest in one bar. One feature far ahead of all others deserves a second look: is it
  known at scoring time, and not the target under another name?
- **The beeswarm**: one line per feature (the same 15), one point per sampled row (at most
  1,500) at its contribution, coloured by the feature's value: blue low, red high, grey null, a
  categorical in one colour (hover for its level). Red on the right means higher values raise
  the score; a wide spread, a strong effect on some rows.

The members of one redundancy cluster share their importance: read them together.

In [ ]:
explanation = views["explanation"]
display(explanation["importance"], explanation["beeswarm"])

## 7. The sign-off view
What a person signs off before the model is registered: the model's final features, with the
evidence to defend each. It is train-only evidence and needs no evaluated run. **This notebook
only shows it: the sign-off itself is recorded in `05_evaluation`**, after the test split's
results, at every automation level. A feature rejected there is an `exclude` and a new run.
- **The status line**: whether this feature list is signed off already, and by which entry. A
  sign-off covers every run of the same data definition with the same final features.
- **`model`**: the final features by out-of-fold contribution, largest first, with
  `contribution_share` and the running `cumulative_share`; `direction_model` (`+`: higher values
  raise the score) beside `direction_eda` and whether they agree; the declared constraint
  (`monotone`) and `wrong_sign` when it goes against EDA's direction; the cluster members it
  `represents`; its `source`; Feature Checks' `notes`; the flags a person kept (`kept_flags`).
- **`held`**: with their open flags; out of training until a person decides.
- **`excluded`**: with the exclude entry in force and its reason.
- **`declared_not_in_model`**: a feature with a monotone constraint that is held, excluded or
  dropped by selection: a constraint protects nothing, a `keep` does.
- **The warnings** under the sections: each constraint against EDA's direction. It holds: a
  person may know more than the window shows.

**Check, feature by feature:** known at scoring time, not the target under another name, a
direction that makes sense (a disagreement with EDA needs a reason). A feature that fails is an
`exclude`, then train again (step 9).

In [ ]:
signoff = views["signoff"]
shown = [signoff["status"], *signoff["sections"].values(), signoff["warnings"]]
display(*[HTML(block) if isinstance(block, str) else block for block in shown])

## 8. The runs so far
**`list_runs`**: the use case's runs on this entity, newest first, one row each: who started it
and when, the data definition, Training's summary (below), Evaluation's once it ran, and the
states `trained`, `evaluated` and `signed_off`. Training's summary: the chosen `family`; the
final `features` and their `feature_signature`; the `trials` run; the out-of-fold metric
(`oof_<metric>`, optimistic: step 5); `recipe_hash` and `model_hash`; the total `seconds`. A run with `trained` false is incomplete: it
crashed, or it is still training. **`show_run`**: one run as plain text: what went in
(`run.json`), what Training did, what Evaluation found, the states and the run folder's files.
The CLI: `python -m core_modelling runs <config> --entity entity_a` and `show-run <config>
--entity entity_a --run <run>`.

In [ ]:
display(list_runs(env, cfg, ENTITY))
print(show_run(env, cfg, ENTITY, result.meta["run_id"]))

## 9. How to iterate
Every run is kept, so iterating is cheap:
1. **Change one thing**, as an entry in the decision audit log (below). None of these needs EDA
   or Feature Checks to rerun.
2. **Train again**: rerun step 1. It writes the next run number; the earlier runs stay.
3. **Compare** in step 8: the out-of-fold metric, the feature count and signature, the seconds.
   A small difference is noise: the out-of-fold metric is optimistic and moves with the folds.
   `05_evaluation` compares two runs on the test split, with intervals.

Paste a call into your own cell after `from core_modelling.config.audit_log import
record_decision` (`CONFIG`, `ENTITY` and `ENVIRONMENT` are the setup cell's). A `set` replaces
the whole value at its path, so list what should stay too.

**A feature back in, or out.** A `keep` of a held feature names the flags it settles, as in
`03_feature_checks` step 9; a `keep` of a feature without an open flag needs no `findings` and
protects it from selection. An `exclude` takes a feature out of every later run.
```python
record_decision(CONFIG, ENTITY, keep=["<feature>"], findings=["<finding>"],
                reason="<why it is known at scoring time>", env_path=ENVIRONMENT)
record_decision(CONFIG, ENTITY, exclude=["<feature>"],
                reason="<why it must not be used>", env_path=ENVIRONMENT)
```

**A quick run, then a main run.** A small budget first shows each family's seconds per trial
and how fast its search settles (step 3); then give the main run its budget per family.
`training.trials` names every listed family and replaces `training.max_iter`:
```python
record_decision(CONFIG, ENTITY, set={"training.max_iter": 20},
                reason="A quick run to time the families", env_path=ENVIRONMENT)
record_decision(CONFIG, ENTITY,
                set={"training.trials": {"lgbm": 100, "xgboost": 40,
                                         "xgb_limitdepth": 40, "catboost": 20}},
                reason="lgbm led the quick run and still improved; catboost trailed",
                env_path=ENVIRONMENT)
```

**Other settings**, the same way: `training.estimator_list`, `training.fold_repeats`,
`training.selection.floor_z` (and `.cluster`, `.prune`), `training.calibration.method`,
`training.monotone` (`{feature: increasing}` or `decreasing`, on numerics and ordered
categoricals only), `training.search_space`, `training.categoricals.min_level_client_count`
and `.unordered`, `training.sampling`. The seed and the decision never change through the log:
the decision is fixed before any result is seen.

**Next: `05_evaluation`.** It scores a run on the test split and the
validation period, writes the model results report, and guides the feature sign-off: at every
automation level a person signs off the model's features there before it is registered.